# Task 2.2 Reproduction of Contribution

**Contribution:** We are reproducing the SVM-CA (Support Vector Machine with Component Analysis) joint optimization algorithm for binary classification, specifically resolving the alternating optimization described in the "Iterative Algorithm" section.

**Evaluation Metric:** Classification error rate (%), same as the tables in the paper.

In [1]:
import numpy as np
from scipy.optimize import minimize

np.random.seed(42)
data = np.load("data/toy_dataset.npz")
K_0 = data["K_0"]
y = data["y"]
X = data["X"]

# Train-test split (80-20)
train_idx = np.arange(160)
test_idx = np.arange(160, 200)

K_train = K_0[np.ix_(train_idx, train_idx)]
y_train = y[train_idx]
y_test = y[test_idx]

We initialize the proxy positive semi-definite matrix. The following block computes standard SVM parameters using convex optimization. It corresponds to Equation (18) solving for $\alpha$ given $V$.

In [2]:
def solve_svm_alpha(K_v, y_tr, C=1.0):
    N = len(y_tr)
    def objective(a):
        return 0.5 * np.dot(a, np.dot(y_tr[:, None] * y_tr[None, :] * K_v, a)) - np.sum(a)
    
    def jacobian(a):
        return np.dot(y_tr[:, None] * y_tr[None, :] * K_v, a) - np.ones(N)
    
    bounds = [(0, C) for _ in range(N)]
    constraints = [{"type": "eq", "fun": lambda a: np.dot(a, y_tr), "jac": lambda a: y_tr}]
    
    a0 = np.zeros(N)
    res = minimize(objective, a0, bounds=bounds, constraints=constraints, jac=jacobian, method="SLSQP", options={"ftol": 1e-6})
    return res.x

Next, we implement the update for the projection matrix $V$. Corresponds to finding top $d$ eigenvectors of $M K_0$.

In [3]:
def update_V(K_0_tr, a, y_tr, rho, d):
    N = len(y_tr)
    ya = y_tr * a
    M = 0.5 * np.outer(ya, ya) + rho * np.eye(N)
    
    val, vec = np.linalg.eig(M @ K_0_tr)
    
    real_indices = np.where(np.isreal(val))[0]
    sorted_idx = real_indices[np.argsort(np.real(val[real_indices]))[::-1]]
    top_d_idx = sorted_idx[:d]
    
    U = np.real(vec[:, top_d_idx])
    
    # V = K_0^{-1} M^{-1} U
    V = np.linalg.solve(K_0_tr, np.linalg.solve(M, U))
    
    V_star = np.zeros_like(V)
    for i in range(d):
        norm_factor = np.sqrt(np.abs(np.dot(V[:, i].T, np.dot(K_0_tr, V[:, i]))))
        if norm_factor > 1e-12:
            V_star[:, i] = V[:, i] / norm_factor
            
    return V_star

We now combine these into the iterative alternating algorithm described in the paper.

In [4]:
# Iterative algorithm loop
C = 1.0
rho = 1.0
d = 2

val0, vec0 = np.linalg.eigh(K_train)
# Clip eigenvalues
pos_idx = val0 > 1e-6
V_init = vec0[:, pos_idx] @ np.diag(1.0 / np.sqrt(val0[pos_idx]))
if V_init.shape[1] > d:
    V_init = V_init[:, -d:]
elif V_init.shape[1] < d:
    d = V_init.shape[1]

V = V_init
for it in range(5):  # 5 iterations max
    K_v = K_train @ V @ V.T @ K_train
    a = solve_svm_alpha(K_v, y_train, C=C)
    V_new = update_V(K_train, a, y_train, rho, d)
    V = V_new

# Final a and K_v
K_v_final = K_train @ V @ V.T @ K_train
a_final = solve_svm_alpha(K_v_final, y_train, C=C)

support_idx = np.where((a_final > 1e-5) & (a_final < C - 1e-5))[0]
ya = y_train * a_final
if len(support_idx) > 0:
    b = y_train[support_idx[0]] - np.sum(ya * K_v_final[:, support_idx[0]])
else:
    b = 0.0

pred_train = np.sign(np.sum(ya[:, None] * K_v_final, axis=0) + b)
err_tr = np.mean(pred_train != y_train)
print(f"Training error rate: {err_tr*100:.2f}%")

np.savez("data/model_params.npz", a=a_final, V=V, b=b, train_idx=train_idx, test_idx=test_idx)

Training error rate: 28.12%
